In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df=pd.read_pickle(f"./df.pkl")
y=df[['dm']]
X=df.drop(columns=['dm'])

In [2]:
X.shape

(13796, 25)

In [3]:
X_trainvalid, X_test, y_trainvalid, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42,
            shuffle=True
        )

In [4]:
X_trainvalid.shape

(11036, 25)

In [5]:
X_train,X_valid, y_train, y_valid=train_test_split(X_trainvalid, y_trainvalid, test_size=0.2, random_state= 65, shuffle=True)

In [6]:
X_valid.shape, y_valid.shape

((2208, 25), (2208, 1))

In [9]:
from recommend import safety_adjust_delta
def apply_safety_adjustment(x, raw_deltas):
    patient = x.iloc[0].to_dict()
    adjusted_deltas = {}

    for var, delta in raw_deltas.items():
        result = safety_adjust_delta(patient, var, float(delta))
        adjusted_delta = float(result["adjusted_delta"])

        if abs(adjusted_delta) > 1e-6:
            adjusted_deltas[var] = adjusted_delta

    return adjusted_deltas

In [10]:
from model.progression_scoring import progression_scoring
from model.optimize_state import compute_total_decision

import itertools
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

epsilons = [1,3,5,10]
lambdas = [0.001,0.005,0.01,0.05,0.1,0.2,0.5,1.0]

results=[]

X_eval = X_valid.sample(100, random_state=42)

BASE_DIR = Path.cwd()

model_paths = [
            str(BASE_DIR / "model" / "final_checkpoints" / f"fold_{i}" / "best-checkpoint-v6.ckpt")
            for i in range(5)
        ]

with open(BASE_DIR / "scalers.pkl", "rb") as f:
    scalers = pickle.load(f)
    
def apply_deltas(X, deltas):
        X_opt = X.copy()

        for col, delta in deltas.items():
            if col in X_opt.columns:
                X_opt[col] = X_opt[col] + delta

        return X_opt


for eps, lam in itertools.product(epsilons,lambdas):

    score_reduction=[]
    l1_changes=[]
    n_changes=[]
    ratio_changes = []
    score_increased = []
    walk_max = []
    empty_recommendation = []

    for idx in range(len(X_eval)):

        x = X_eval.iloc[[idx]]

        before = progression_scoring(
            x,
            model_paths,
            scalers
        )

        raw_deltas = compute_total_decision(
            x,
            model_paths,
            scalers,
            epsilon=eps,
            lambda_reg=lam
        )
        
        deltas=apply_safety_adjustment(x,raw_deltas)

        x_after = apply_deltas(x,deltas)

        after = progression_scoring(
            x_after,
            model_paths,
            scalers
        )
        
        inv = {
        'wk_smk': (0.0, 420.0),
        'wk_alc': (0.0, 40.0),
        'wk_mvpa_play': (0.0, 300.0),
        'wk_walk': (0.0, 1260.0),
        'wk_sleep': (360.0, 540.0),
        'stress': (1.0, 4.0),
        'wk_break': (0.0, 6.0),
        'wk_lunch': (0.0, 6.0),
        'wk_dinner': (0.0, 6.0),
        'wk_veg1': (0.0, 21.0),
        'wk_veg2': (0.0, 21.0),
        'wk_fruit': (0.0, 21.0),
        }

        ratios=[]
        for var, delta in deltas.items():

            delta = float(delta)
            min_val, max_val = inv[var]
            scale = max_val - min_val

            ratio = abs(delta) / scale
            ratios.append(ratio)

        score_reduction.append(before-after)
        
        score_increased.append(after > before)

        l1_changes.append(
                np.sum(np.abs(list(deltas.values())))
            )

        n_changes.append(len(deltas))

        ratio_changes.append(np.mean(ratios))
        
        empty_recommendation.append(len(deltas) == 0)

        # walk가 상한(1260)에 도달했는지
        walk_max.append(
            x_after.iloc[0]["wk_walk"] >= 1260 - 1e-6
        )

    results.append({

        "epsilon":eps,
        "lambda":lam,

        "score_reduction":
        np.mean(score_reduction),

        "L1_change":
        np.mean(l1_changes),

        "n_modified":
        np.mean(n_changes),

        "mean_ratio_change": np.mean(ratio_changes),
        
        "score_increase_rate": np.mean(score_increased),
        
        "walk_max_rate": np.mean(walk_max),

        "empty_recommendation_rate": np.mean(empty_recommendation)

    })

results=pd.DataFrame(results)

c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:56: The loaded checkpoint was produced with Lightning v2.5.2, which is newer than your current Lightning version: v2.5.1.post0
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:56: The loaded checkpoint was produced with Lightning v2.5.2, which is newer than your current Lightning version: v2.5.1.post0
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightning\p

KeyboardInterrupt: 

In [ ]:
results

,epsilon,lambda,score_reduction,L1_change,n_modified,mean_ratio_change
0,1,0.001,0.606870,126.9034,9.14,0.065924
1,1,0.005,0.602012,129.0891,9.13,0.065079
2,1,0.010,0.592534,126.5732,9.12,0.063729
3,1,0.050,0.532674,113.9443,8.83,0.054597
4,1,0.100,0.473126,97.9059,8.48,0.046212
5,1,0.200,0.375588,65.6323,7.59,0.038682
6,1,0.500,0.128265,27.4377,5.82,0.018681
7,1,1.000,-0.000159,18.7194,4.90,0.007855
8,3,0.001,1.402936,353.5019,9.42,0.155283
9,3,0.005,1.394352,350.3374,9.41,0.154849
